In [1]:
!pip install gradio==4.41.0 reportlab==4.0.7 autopep8==2.1.0


  Using cached gradio-4.41.0-py3-none-any.whl.metadata (15 kB)
  Using cached gradio_client-1.3.0-py3-none-any.whl.metadata (7.1 kB)
Using cached gradio-4.41.0-py3-none-any.whl (12.6 MB)
Using cached gradio_client-1.3.0-py3-none-any.whl (318 kB)
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 2.2.0
    Uninstalling gradio_client-2.2.0:
      Successfully uninstalled gradio_client-2.2.0
  Attempting uninstall: gradio
    Found existing installation: gradio 6.8.0
    Uninstalling gradio-6.8.0:
      Successfully uninstalled gradio-6.8.0


In [2]:
!pip uninstall -y gradio
!pip install gradio reportlab autopep8


Found existing installation: gradio 4.41.0
Uninstalling gradio-4.41.0:
  Successfully uninstalled gradio-4.41.0
  Using cached gradio-6.8.0-py3-none-any.whl.metadata (16 kB)
  Using cached gradio_client-2.2.0-py3-none-any.whl.metadata (7.1 kB)
Using cached gradio-6.8.0-py3-none-any.whl (24.2 MB)
Using cached gradio_client-2.2.0-py3-none-any.whl (58 kB)
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 1.3.0
    Uninstalling gradio_client-1.3.0:
      Successfully uninstalled gradio_client-1.3.0


In [3]:
import ast
import gradio as gr
import autopep8
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Preformatted
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from datetime import datetime


# ==============================
# Core Analyzer Class
# ==============================

class DocGenieAnalyzer:

    @staticmethod
    def extract_function_signature(code):
        tree = ast.parse(code)

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                params = []

                for arg in node.args.args:
                    param_type = "Any"
                    if arg.annotation:
                        param_type = ast.unparse(arg.annotation)

                    params.append({
                        "name": arg.arg,
                        "type": param_type
                    })

                return_type = "Any"
                if node.returns:
                    return_type = ast.unparse(node.returns)

                return {
                    "name": node.name,
                    "params": params,
                    "return_type": return_type
                }

        return None


    @staticmethod
    def analyze_function_logic(code):
        tree = ast.parse(code)

        has_loops = False
        has_conditions = False
        operations = []

        for node in ast.walk(tree):

            if isinstance(node, (ast.For, ast.While)):
                has_loops = True

            if isinstance(node, ast.If):
                has_conditions = True

            if isinstance(node, ast.BinOp):
                if isinstance(node.op, ast.Add):
                    operations.append("addition")
                elif isinstance(node.op, ast.Sub):
                    operations.append("subtraction")
                elif isinstance(node.op, ast.Mult):
                    operations.append("multiplication")
                elif isinstance(node.op, ast.Div):
                    operations.append("division")

        description = "Performs computation."

        if has_loops:
            description += " Includes loop operations."

        if has_conditions:
            description += " Uses conditional logic."

        if operations:
            description += " Applies " + ", ".join(set(operations)) + "."

        return description


    @staticmethod
    def generate_google_docstring(signature, description):

        doc = f'"""{signature["name"].replace("_", " ").title()}.\n\n'
        doc += description + "\n\n"

        doc += "Args:\n"
        for param in signature["params"]:
            doc += f'    {param["name"]} ({param["type"]}): Description of {param["name"]}.\n'

        doc += "\nReturns:\n"
        doc += f'    {signature["return_type"]}: Description of return value.\n'
        doc += '"""'

        return doc


    @staticmethod
    def generate_numpy_docstring(signature, description):

        doc = f'"""{signature["name"].replace("_", " ").title()}.\n\n'
        doc += description + "\n\n"

        doc += "Parameters\n----------\n"
        for param in signature["params"]:
            doc += f'{param["name"]} : {param["type"]}\n'
            doc += f'    Description of {param["name"]}.\n'

        doc += "\nReturns\n-------\n"
        doc += f'{signature["return_type"]}\n'
        doc += "    Description of return value.\n"
        doc += '"""'

        return doc


# ==============================
# Main Processing Function
# ==============================

def generate_docstring(code, style):

    try:
        signature = DocGenieAnalyzer.extract_function_signature(code)

        if not signature:
            return "No function found in the provided code."

        description = DocGenieAnalyzer.analyze_function_logic(code)

        if style == "google":
            docstring = DocGenieAnalyzer.generate_google_docstring(signature, description)
        else:
            docstring = DocGenieAnalyzer.generate_numpy_docstring(signature, description)

        formatted_code = autopep8.fix_code(code)

        final_output = formatted_code.replace(
            f"def {signature['name']}",
            f"def {signature['name']}\n    {docstring}"
        )

        return final_output

    except Exception as e:
        return f"Error: {str(e)}"


# ==============================
# PDF Export
# ==============================

def export_pdf(code_output):

    file_path = "DocGenie_Output.pdf"
    doc = SimpleDocTemplate(file_path, pagesize=letter)
    elements = []
    styles = getSampleStyleSheet()

    elements.append(Paragraph("<b>Doc-Genie Generated Documentation</b>", styles["Title"]))
    elements.append(Spacer(1, 0.5 * inch))
    elements.append(Preformatted(code_output, styles["Code"]))

    doc.build(elements)

    return file_path


# ==============================
# Gradio Interface
# ==============================

with gr.Blocks(title="Doc-Genie") as demo:

    gr.Markdown("# 📘 Doc-Genie - Python Docstring Generator")

    with gr.Row():

        with gr.Column():
            code_input = gr.Code(label="Enter Python Function", language="python")
            style_selector = gr.Radio(["google", "numpy"], value="google", label="Docstring Style")
            generate_btn = gr.Button("Generate Docstring")

        with gr.Column():
            code_output = gr.Code(label="Generated Output", language="python")
            pdf_btn = gr.Button("Download PDF")
            pdf_file = gr.File()

    generate_btn.click(generate_docstring, inputs=[code_input, style_selector], outputs=code_output)
    pdf_btn.click(export_pdf, inputs=code_output, outputs=pdf_file)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a13a08168376273d47.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
